In [3]:
!pip install pyspark

Q1 - Roles of Driver, Cluster Manager and Executor

Driver:
The Driver is the master of the Spark application. It runs
the main program, creates the SparkSession, converts our
code into a DAG (execution plan), and sends tasks to the
executors. If the Driver fails, the whole job fails.

Cluster Manager:
The Cluster Manager manages the resources of the cluster.
When the Driver needs machines to run tasks, it asks the
Cluster Manager. It then allocates worker nodes. Examples
are YARN, Kubernetes, and Standalone.

Executor:
Executors are the worker processes that run on each machine.
They receive tasks from the Driver, execute them on their
data partition, and send results back. If one Executor
fails, Spark automatically restarts it.

Q2 - Lazy Evaluation in Spark

Lazy Evaluation means when we write transformations in Spark,
it does not run them immediately. It just saves the steps and
makes a plan. This plan is called DAG.

Spark waits until we call an action like show() or count().
Only then it looks at the full plan and runs everything
together in an optimized way.

For example:
df.filter(col("category") == "Electronics")  
→ nothing happens yet

df.select("product_id", "price")             
→ still nothing happens

df.show()  
→ now Spark runs all the above steps at once

This is useful because Spark can look at all the steps
together and skip unnecessary work. So it saves time
and processes only what is actually needed.

In [4]:
#sparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName("Week6_Spark_assignment")\
        .getOrCreate()
print("Spark is ready!")
print("spark version:",spark.version)

Spark is ready!
spark version: 4.0.3


In [5]:
import os

os.makedirs("data", exist_ok=True)
with open("data/source.csv", "w") as f:
    f.write("product_id,old_name,base_price,price,category\n")
    f.write("P001,Widget A,100.0,19.99,Electronics\n")
    f.write("P002,Widget B,200.0,5.50,Clothing\n")
    f.write("P003,Widget C,500.0,99.99,Electronics\n")
    f.write("P004,Widget D,150.0,30.00,Books\n")
    f.write("P005,Widget E,800.0,149.99,Electronics\n")
    f.write("P006,Widget F,50.0,9.99,Clothing\n")

with open("data/orders.csv", "w") as f:
    f.write("order_id,user_id,status,amount\n")
    f.write("O001,U1,Completed,1500\n")
    f.write("O002,U2,Pending,200\n")
    f.write("O003,U3,Completed,800\n")
    f.write("O004,,Completed,2000\n")
    f.write("O005,U5,Completed,1200\n")
    f.write("O006,U6,Cancelled,3000\n")

print("Files in data folder:", os.listdir("data"))



Files in data folder: ['orders.csv', 'source.csv']


In [20]:
#Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

df = spark.read\
     .option("header","true")\
     .option("inferSchema","true") \
     .csv("data/source.csv")
df.show()
df.printSchema()

+----------+--------+----------+------+-----------+
|product_id|old_name|base_price| price|   category|
+----------+--------+----------+------+-----------+
|      P001|Widget A|     100.0| 19.99|Electronics|
|      P002|Widget B|     200.0|   5.5|   Clothing|
|      P003|Widget C|     500.0| 99.99|Electronics|
|      P004|Widget D|     150.0|  30.0|      Books|
|      P005|Widget E|     800.0|149.99|Electronics|
|      P006|Widget F|      50.0|  9.99|   Clothing|
+----------+--------+----------+------+-----------+

root
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- category: string (nullable = true)



Q4)What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

answer)
CSV is row-based storage.It stores data row by row,so spark must scan every row even if we only need one column.

Parquet is columnar storage.it stores data by column,so spark reads only the required column,skipping everything else.

parquet is better for performance because:
-It reads onlly required columns
-better compression ratio
-supports predicate pushdown


In [7]:
from pyspark.sql.functions import col
# Q5 - Select product_id and price where category is Electronics

result_q5 = df.filter(col("category") == "Electronics") \
              .select("product_id","price")
result_q5.show()



+----------+------+
|product_id| price|
+----------+------+
|      P001| 19.99|
|      P003| 99.99|
|      P005|149.99|
+----------+------+



In [8]:
# Q6 - Rename old_name to new_name and cast price to Double
result_q6 = df.withColumnRenamed("old_name","product_name")\
              .withColumn("price",col("price").cast("Double"))
result_q6.show()
result_q6.printSchema()

+----------+------------+----------+------+-----------+
|product_id|product_name|base_price| price|   category|
+----------+------------+----------+------+-----------+
|      P001|    Widget A|     100.0| 19.99|Electronics|
|      P002|    Widget B|     200.0|   5.5|   Clothing|
|      P003|    Widget C|     500.0| 99.99|Electronics|
|      P004|    Widget D|     150.0|  30.0|      Books|
|      P005|    Widget E|     800.0|149.99|Electronics|
|      P006|    Widget F|      50.0|  9.99|   Clothing|
+----------+------------+----------+------+-----------+

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- category: string (nullable = true)



Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

answer)
Apache Spark provides fault tolerance using a Lineage Graph, also known as a Directed Acyclic Graph (DAG). Instead of storing multiple copies of intermediate data, Spark keeps track of all the transformations (such as map, filter, and reduce) that are applied to the original data. This sequence of transformations is called the lineage of an RDD.

When a worker node fails, the data stored on that node is lost. Spark does not need to restart the entire application. Instead, it refers to the Lineage Graph to identify how the lost partition was created. It then recomputes only the missing partition by applying the same transformations to the original data. The remaining partitions, which are still available on other worker nodes, are not affected.

This approach makes Spark highly fault tolerant while reducing storage overhead because it does not rely on replicating intermediate data. By recomputing only the lost data, Spark ensures efficient recovery and continues processing with minimal interruption.


In [9]:
#read orders data
df_orders = spark.read\
     .option("header","true")\
     .option("inferSchema","true")\
     .csv("data/orders.csv")
df_orders.show()

+--------+-------+---------+------+
|order_id|user_id|   status|amount|
+--------+-------+---------+------+
|    O001|     U1|Completed|  1500|
|    O002|     U2|  Pending|   200|
|    O003|     U3|Completed|   800|
|    O004|   NULL|Completed|  2000|
|    O005|     U5|Completed|  1200|
|    O006|     U6|Cancelled|  3000|
+--------+-------+---------+------+



In [10]:
#Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.
result_q8 = df_orders.filter(
    (col("status")=="Completed") & (col("amount")>1000))
result_q8.show()


+--------+-------+---------+------+
|order_id|user_id|   status|amount|
+--------+-------+---------+------+
|    O001|     U1|Completed|  1500|
|    O004|   NULL|Completed|  2000|
|    O005|     U5|Completed|  1200|
+--------+-------+---------+------+



Q9)Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

answer)
Predicate Pushdown is an optimization technique in Spark where
filter conditions are pushed down to the storage level BEFORE
data is loaded into memory.

Parquet files store metadata (min/max values) for each column chunk.
When Spark has a filter like amount > 1000, it checks the Parquet
metadata first. If a chunk's maximum value is 500, Spark skips
that entire chunk without reading it.

Effect on memory:
- Without Predicate Pushdown → ALL data loads into memory, THEN filter runs
- With Predicate Pushdown → only matching data loads into memory

This reduces the amount of data read from disk and makes
queries much faster on large datasets



In [11]:
#Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).
result_q10 = df.withColumn("final_price",col("base_price")*1.18)

result_q10.show()

+----------+--------+----------+------+-----------+-----------+
|product_id|old_name|base_price| price|   category|final_price|
+----------+--------+----------+------+-----------+-----------+
|      P001|Widget A|     100.0| 19.99|Electronics|      118.0|
|      P002|Widget B|     200.0|   5.5|   Clothing|      236.0|
|      P003|Widget C|     500.0| 99.99|Electronics|      590.0|
|      P004|Widget D|     150.0|  30.0|      Books|      177.0|
|      P005|Widget E|     800.0|149.99|Electronics|      944.0|
|      P006|Widget F|      50.0|  9.99|   Clothing|       59.0|
+----------+--------+----------+------+-----------+-----------+



Q11: What is the difference between Transformations and Actions? Provide two examples of each.

answer)
Transformations are LAZY operations ,they do not execute immediately.
They just build a plan (DAG). Spark waits until an Action is called.

Examples of Transformations:
1. filter()  - filters rows based on condition
2. select()  - selects specific columns

Actions are EAGER operations  they trigger actual execution of
all the transformations built so far.

Examples of Actions:
1. show()  - displays the result on screen
2. count() - counts total number of rows

In [12]:
df_orders.write.mode("overwrite").parquet("data/orders_parquet")
print("parquest file created!")

parquest file created!


In [13]:
#Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

df_parquet = spark.read.parquet("data/orders_parquet")
df_clean = df_parquet.filter(col("user_id").isNotNull())
df_clean.write.option("header","true").mode("overwrite").csv("data/output")

print("pipeline complete!")
df_clean.show()

pipeline complete!
+--------+-------+---------+------+
|order_id|user_id|   status|amount|
+--------+-------+---------+------+
|    O001|     U1|Completed|  1500|
|    O002|     U2|  Pending|   200|
|    O003|     U3|Completed|   800|
|    O005|     U5|Completed|  1200|
|    O006|     U6|Cancelled|  3000|
+--------+-------+---------+------+



Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?
Client Mode:
- The Driver runs on your local machine (where you submit the job)
- You must stay connected until the job finishes
- Best for development and testing

Cluster Mode:
- The Driver runs inside the cluster on a worker node
- You can disconnect after submitting the job
- Best for production jobs

In [15]:
from pyspark.sql import Row

data =[
    ("North","High"),
    ("South","Low"),
    ("East",  "High"),
    ("North", "Low"),
    ("West",  "Medium"),
]
df_q14 = spark.createDataFrame(data,["region","priority"])
df_q14.show()

+------+--------+
|region|priority|
+------+--------+
| North|    High|
| South|     Low|
|  East|    High|
| North|     Low|
|  West|  Medium|
+------+--------+



In [17]:
result_q14 = df_q14.filter(
    (col("region")=="North") |(col("priority")=="High")
)
result_q14.show()


+------+--------+
|region|priority|
+------+--------+
| North|    High|
|  East|    High|
| North|     Low|
+------+--------+



Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

When we use collect(), it brings all the data from all the
machines back to one single machine (the Driver). If the
dataset is very large like terabytes of data, the Driver
cannot handle that much data at once and it will crash
with memory error.

But show(5) only fetches 5 rows to show on screen. The
rest of the data stays on the worker machines. So the
Driver does not get overloaded.

That is why it is always safer to use show(5) when we
just want to check or explore the data, and only write
to disk when we want to save all results.